In [1]:
from dotenv import load_dotenv
load_dotenv()
import os
from supabase import create_client, Client
supabase_url = os.environ.get("SUPABASE_URL")
supabase_api_key = os.environ.get("SUPBASE_KEY")

supabase: Client = create_client(supabase_url, supabase_api_key)

In [2]:
from openai import OpenAI
client = OpenAI(api_key = os.getenv("OPENAI_API_KEY"))

In [6]:
def workflow_research(date, next_date):
    result = supabase.table('curation_selected_items')\
        .select('event_id, output,sources, news_date','topic')\
    .gte('created_at', date)\
    .lt('created_at', next_date)\
    .eq('topic', 'Workflow improvement')\
        .order('created_at', desc=False)\
        .execute()
    
    return result.data

In [7]:
workflow_deep_dive = workflow_research('2026-02-24','2026-02-25')

In [8]:
workflow_deep_dive

[{'event_id': '1391_2026-02-23',
  'output': 'Leonardo.Ai announced a refreshed brand identity and launched a Creative Engine API after joining Canva, positioning the product as a developer- and enterprise-ready foundation for integrating generative-creative models and workflows into third-party platforms. Practical impact: enables product and engineering teams to embed generative visual capabilities into apps and pipelines, shifting some creative work from end-user tools into programmatic automation and platform-level integrations.',
  'sources': [{'url': 'https://www.edtechinnovationhub.com/news/leonardoai-launches-new-brand-and-creative-engine-api-under-canva-ownership',
    'name': 'EdTech Innovation Hub'}],
  'news_date': '2026-02-23',
  'topic': 'Workflow improvement'},
 {'event_id': '1403_2026-02-23',
  'output': 'Wispr Flow launched its Android app for AI-powered dictation (published Feb 23, 2026). The release adds a floating-bubble Android UI, supports dictation across apps, c

In [9]:
def openai_research_workflow(research_input):
    for i in research_input:
        response = client.responses.create(
            model = "gpt-5-nano",
            tools = [{
                "type": "web_search"
            }],
            include = ["web_search_call.action.sources"],
            input = f"""
            You are a senior research analyst for Krux.

            TASK:
              Create a deep, fact-checked brief for ONE AI report/paper/benchmark release.
            
            GOAL:
              Extract the highest-value findings and implementation guidance so a later 100-word summary can capture most practical signal.
            
            OUTPUT FORMAT:
              - 12 to 16 bullet points.
              - Each bullet must include inline citations:
                [Source: <publisher>, <url>]
              - No text before or after bullets.
            
            MANDATORY STRUCTURE:
              1) REPORT SNAPSHOT
              - What was published, by whom, and when.
              - Scope: geography, sectors, sample size, time window, method type.
            
              2) CORE FINDINGS (MOST IMPORTANT)
              - 4 to 6 bullets with the strongest quantified findings.
              - Include exact numbers, not vague language.
            
              3) WHAT THIS ACTUALLY MEANS
              - 2 to 3 bullets translating findings into plain English implications for real teams.
            
              4) IMPLEMENTATION PLAYBOOK
              - 3 to 4 bullets: what organizations should do in the next quarter.
              - Include sequencing (e.g., data readiness -> pilot -> measurement -> rollout).
            
              5) METRICS TO TRACK
              - 1 or 2 bullets on KPIs teams should monitor to apply the report in practice.
            
              6) LIMITATIONS / CAVEATS
              - 1 or 2 bullets on methodology limits, sample bias, missing data, or uncertainty.
            
              7) DECISION TAKEAWAY
              - 1 bullet: “If a team only does one thing based on this report, it should be X.”
            
              STRICT RULES:
              - No hype, no opinion, no generic AI commentary.
              - Every factual claim must be source-backed.
              - If a key detail is missing, write: "Not disclosed in cited sources."
              - Prefer primary sources (original report/paper) over secondary articles.
              - If secondary coverage conflicts with primary report, prioritize primary and note conflict.
            
              QUALITY CHECK:
              - Must contain quantified findings.
              - Must contain actionable implementation steps.
              - Must clearly separate findings from interpretation.
              - Must include limitations.
            
              Report/event to research:
              {i['output']}
            """,
        )

        output = response.output_text
        print(output)

        final_dictionary = {
            'event_id': i['event_id'],
            'news_date': i['news_date'],
            'output': output,
            'model_provider': 'openai',
            'topic': i['topic']
        }
        print(final_dictionary)

        save_research(final_dictionary)

        print("✅ Done")

In [10]:
def save_research(research_json):
    supabase.table('research_assistant').insert({
        'event_id': research_json['event_id'],
        'model_provider': research_json['model_provider'],
        'news_date': research_json['news_date'],
        'output': research_json['output'],
        'topic': research_json['topic']
    }).execute()

In [11]:
openai_research_workflow(workflow_deep_dive[0:])

- REPORT SNAPSHOT — Leonardo.Ai published “Creative Engine API and New Brand Identity After Joining Canva” on February 23, 2026, authored by adminlnd; context notes Canva’s acquisition of Leonardo.Ai publicly announced July 30, 2024; scope cited includes global developers/enterprises, Leonardo’s Phoenix foundational model, and the AI API/branding rollout as the primary deliverables. [Source: Leonardo.Ai, https://leonardoaiapp.com/leonardo-ai-unveils-creative-engine-api-and-new-brand-identity-after-joining-canva], [Source: Canva, https://www.canva.com/newsroom/news/leonardo-ai/], [Source: The Verge, https://www.theverge.com/2024/7/30/24209421/canva-leonardo-generative-ai-platform-acquisition-design-software]

- CORE FINDING 1 — Public Creative Engine API launched February 23, 2026, enabling developers and businesses to connect Leonardo’s generative AI models directly into their own apps and platforms, enabling use cases in design, media, gaming, and marketing. [Source: Leonardo.Ai, http